# Data Generation on Apple Silicon

In [1]:
import importlib.metadata
import time
import json
from pathlib import Path

import torch
from tqdm import tqdm

In [2]:
# -----------------------------
# Environment checks
# -----------------------------

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    from mlx_lm import load, generate
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

Using device: mps


In [3]:
# -----------------------------
# Configuration
# -----------------------------
if torch.cuda.is_available():
    MLX_MODEL = "lmstudio-community/Qwen3-4B-Instruct-2507-GGUF"
else:
    MLX_MODEL = "lmstudio-community/Qwen3-4B-Instruct-2507-MLX-8bit"

print(f"Using model: {MLX_MODEL}")

NUM_PARAPHRASES = 1          # exactly one paraphrase per chunk
CHUNK_SIZE_TOKENS = 4096     # ~full context window
CHUNK_OVERLAP_TOKENS = 256   # small overlap for boundary safety
MAX_TOKENS = CHUNK_SIZE_TOKENS      # paraphrase length

INPUT_CORPUS = "Datasets/finance_bench_corpus.json"
OUTPUT_DIR = Path("paraphrase_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Using model: lmstudio-community/Qwen3-4B-Instruct-2507-MLX-8bit


In [4]:
PARAPHRASE_PROMPT_TEMPLATE = """Use the information in the following snippet to write an informational paragraph in your own words.
Make sure to cover all the information, including all entities, dates and places in the original document.
Do not add additional material.
Directly output the paragraph and nothing else.

<document>
{doc}
</document>
"""

In [5]:
def mlx_generate_compat(model, tokenizer, prompt: str, max_tokens: int) -> str:
    """
    Call mlx_lm.generate with chat-template support.
    Uses DEFAULT decoding parameters.
    """
    if tokenizer.chat_template is not None:
        messages = [{"role": "user", "content": prompt}]
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_dict=False,
        )

    return generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=max_tokens,
        verbose=False,
    )

In [6]:
def chunk_text(tokenizer, text, chunk_size, overlap):
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0

    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunks.append(tokenizer.decode(chunk_tokens))
        start = end - overlap
        if start < 0:
            start = 0

    return chunks

In [7]:
def get_last_completed_chunk(path):
    """
    Returns the highest chunk_id already written to a JSONL file.
    If file doesn't exist, returns -1.
    """
    if not path.exists():
        return -1

    last_chunk = -1
    with open(path, "r") as f:
        for line in f:
            try:
                obj = json.loads(line)
                last_chunk = max(last_chunk, obj.get("chunk_id", -1))
            except json.JSONDecodeError:
                continue
    return last_chunk

In [8]:
with open(INPUT_CORPUS, "r") as f:
    corpus = json.load(f)

print(f"Loaded corpus with {len(corpus)} documents")


Loaded corpus with 24 documents


In [9]:
print("Loading model...")
model, tokenizer = load(MLX_MODEL)
print("Model loaded.")

Loading model...


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Model loaded.


In [10]:
doc_pbar = tqdm(corpus, desc="Documents")
for doc_idx, entry in enumerate(doc_pbar):
    doc_name = entry.get("doc_name", f"doc_{doc_idx}")
    full_doc = entry["text"]

    chunks = chunk_text(
        tokenizer,
        full_doc,
        CHUNK_SIZE_TOKENS,
        CHUNK_OVERLAP_TOKENS,
    )

    doc_pbar.set_postfix_str(f"{doc_name} ({len(chunks)} chunks)")

    output_path = OUTPUT_DIR / f"{doc_name}.jsonl"
    last_chunk = get_last_completed_chunk(output_path)
    start_chunk = last_chunk + 1
    
    if last_chunk >= 0:
        tqdm.write(f"Resuming {doc_name} from chunk {start_chunk}")

    with open(output_path, "a") as out_f:
        for chunk_idx in tqdm(range(start_chunk, len(chunks)), desc=f"Chunks", leave=False):
            chunk = chunks[chunk_idx]
            prompt = PARAPHRASE_PROMPT_TEMPLATE.format(doc=chunk)

            start_time = time.time()
            paraphrase = mlx_generate_compat(
                model,
                tokenizer,
                prompt,
                max_tokens=MAX_TOKENS,
            )
            elapsed = time.time() - start_time

            num_tokens = len(tokenizer.encode(paraphrase))

            record = {
                "doc_name": doc_name,
                "chunk_id": chunk_idx,
                "method": "paraphrase",
                "text": paraphrase,
                "tokens": num_tokens,
                "generation_time_sec": elapsed,
            }

            out_f.write(json.dumps(record) + "\n")
            out_f.flush()

    tqdm.write(f"Saved {doc_name} to {output_path}")

Documents:   0%|          | 0/24 [00:00<?, ?it/s, 3M_2018_10K (37 chunks)]

Resuming 3M_2018_10K from chunk 13


Documents:   0%|          | 0/24 [02:53<?, ?it/s, 3M_2018_10K (37 chunks)]


KeyboardInterrupt: 

In [ ]:
SYNTHETIC_QA_PROMPT = """Generate a comprehensive list of fact-based questions and corresponding answers that can be answered explicitly from the document.

Requirements:
- Cover all entities, including people, organizations, dates, locations, quantities, and named concepts.
- Questions must be unambiguous, properly capitalized, and end with a question mark.
- Answers must be as concise as possible and use wording from the document when applicable.
- Output ONE question-answer pair per line.
- Separate the question and answer by a single space.
- Do NOT add any commentary or extra text.
- Do NOT invent information not present in the document.

<document>
{chunk}
</document>
"""

In [ ]:
def generate_synthetic_qa(chunk, max_tokens=1024):
    prompt = SYNTHETIC_QA_PROMPT.format(chunk=chunk)
    text = mlx_generate_compat(model, tokenizer, prompt, max_tokens)
    return [l.strip() for l in text.split("\n") if l.strip()]

In [ ]:
def parse_qa_lines(lines):
    """
    Parse alternating question / answer lines.
    Returns list of (question, answer).
    """
    pairs = []
    i = 0
    while i + 1 < len(lines):
        q = lines[i].strip()
        a = lines[i + 1].strip()

        if q.endswith("?") and a:
            pairs.append((q, a))

        i += 2
    return pairs

In [ ]:
import os

QA_OUTPUT_DIR = Path("synthetic_qa_outputs")
QA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

doc_pbar = tqdm(corpus, desc="[QA] Documents")
for doc_idx, entry in enumerate(doc_pbar):
    doc_name = entry.get("doc_name", f"doc_{doc_idx}")
    full_doc = entry["text"]

    chunks = chunk_text(
        tokenizer,
        full_doc,
        CHUNK_SIZE_TOKENS,
        CHUNK_OVERLAP_TOKENS,
    )

    doc_pbar.set_postfix_str(f"{doc_name} ({len(chunks)} chunks)")

    output_path = QA_OUTPUT_DIR / f"{doc_name}.jsonl"
    last_chunk = get_last_completed_chunk(output_path)
    start_chunk = last_chunk + 1
    
    if last_chunk >= 0:
        tqdm.write(f"[QA] Resuming {doc_name} from chunk {start_chunk}")

    with open(output_path, "a") as out_f:
        for chunk_idx in tqdm(range(start_chunk, len(chunks)), desc=f"Chunks", leave=False):
            chunk = chunks[chunk_idx]
            qa_lines = generate_synthetic_qa(chunk)
            pairs = parse_qa_lines(qa_lines)

            for question, answer in pairs:
                record = {
                    "doc_name": doc_name,
                    "chunk_id": chunk_idx,
                    "question": question,
                    "answer": answer,
                    "method": "synthetic_qa",
                }

                out_f.write(json.dumps(record) + "\n")
                out_f.flush()
                os.fsync(out_f.fileno())

    tqdm.write(f"[QA] Saved {doc_name} to {output_path}")

In [ ]:
D31_STRATEGY_PROMPT = """Consider the following document. 
What are some strategies specific to this document that I can use to help me learn and remember all of the information contained?

Requirements:
- Strategies should be specific to the structure and content of the document.
- Use markdown.
- Prefix each strategy with ##.
- Do NOT summarize the document.
- Do NOT repeat the document content verbatim.

<document>
{chunk}
</document>
"""

In [ ]:
def generate_task_agnostic_strategies(chunk, max_tokens=512):
    prompt = D31_STRATEGY_PROMPT.format(chunk=chunk)
    text = mlx_generate_compat(model, tokenizer, prompt, max_tokens)
    return text.strip()

In [ ]:
ACTIVE_READING_PROMPT = """Here is a learning strategy:

{strategy}

Apply this strategy to the following document:

<document>
{chunk}
</document>
"""

In [ ]:
def apply_active_reading(strategy, chunk, max_tokens=1024):
    prompt = ACTIVE_READING_PROMPT.format(
        strategy=strategy,
        chunk=chunk
    )
    return mlx_generate_compat(model, tokenizer, prompt, max_tokens)

In [ ]:
import os
from pathlib import Path

ACTIVE_READING_DIR = Path("active_reading_outputs")
ACTIVE_READING_DIR.mkdir(parents=True, exist_ok=True)

# ---- pick ONE document ----
doc_idx = 0
entry = corpus[0]

doc_name = entry.get("doc_name", f"doc_{doc_idx}")
full_doc = entry["text"]

# ---- chunk document ----
chunks = chunk_text(
    tokenizer,
    full_doc,
    CHUNK_SIZE_TOKENS,
    CHUNK_OVERLAP_TOKENS,
)

tqdm.write(f"[AR] {doc_name}: {len(chunks)} chunks")

# ---- output file ----
output_path = ACTIVE_READING_DIR / f"{doc_name}.jsonl"
last_chunk = get_last_completed_chunk(output_path)
start_chunk = last_chunk + 1

if last_chunk >= 0:
    tqdm.write(f"[AR] Resuming from chunk {start_chunk}")

with open(output_path, "a") as out_f:
    for chunk_idx in tqdm(range(start_chunk, len(chunks)), desc=f"[AR] Chunks ({doc_name})"):
        chunk = chunks[chunk_idx]

        # ---- D.3.1: generate strategies ----
        strategy_text = generate_task_agnostic_strategies(chunk)

        # ---- D.3: apply strategies ----
        active_reading_output = apply_active_reading(strategy_text, chunk)

        record = {
            "doc_name": doc_name,
            "chunk_id": chunk_idx,
            "strategy_type": "task_agnostic",
            "strategy": strategy_text,
            "active_reading": active_reading_output,
            "method": "active_reading_d3",
        }

        out_f.write(json.dumps(record) + "\n")
        out_f.flush()
        os.fsync(out_f.fileno())

tqdm.write(f"[AR] Saved all chunks to {output_path}")